# `Summary`
- `1.`The notebook loads your configuration and API keys so it can talk to external AI services.
- `2.`It goes to the Educosys GenAI course web page and pulls the page content into the notebook.
- `3.`That big page is broken into many small, readable pieces so they can be searched and reused easily.
- `4.`All those pieces are stored in a special “searchable memory” so the system can quickly find the most relevant parts when you ask a question.
- `5.`A question‑answering pattern (the rlm/rag-prompt style) is set up, telling the AI:
“Here is some course information (context) and a user question. Answer only from this information, briefly.”

- `6.`A chat AI model (first OpenAI, later Gemini) is connected to this setup so it can generate answers.

- `7.`A pipeline (the RAG chain) is created:
    - Take your question,
    - Find the most relevant pieces of the Educosys page,
    - Plug them and your question into the Q&A prompt,
    - Let the AI write an answer.
- `8.`A small helper is added to print out the full prompt sent to the AI, so you can see exactly what context and question it is using when it answers.

Finally, you ask natural‑language questions like “Who is the tutor?” or “What is the course about?” and the system answers using only information pulled from the Educosys course page.  https://app.eraser.io/workspace/9mNYQUL0NnOyQlWsoSPz

In [1]:
# pip install -U langchain_core langchain-community langchain-google-genai chromadb python-dotenv

In [2]:
# Extracting information from the document and must answer any question related to the document

In [3]:
# import os
# from dotenv import load_dotenv

# load_dotenv()  # loads variables from 0.00341 USD.env into os.environ

# os.environ["LANGCHAIN_API_KEY"] = userdata.get('langchainKey')
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "educosysGenAIClass"

In [4]:
from dotenv import load_dotenv
load_dotenv()

import os
os.getenv("OPENAI_API_KEY")
os.getenv("LANGCHAIN_TRACING_V2")
os.getenv("LANGCHAIN_API_KEY")
os.getenv("LANGSMITH_PROJECT")

'educosysGenAIClass'

In [5]:
import os

os.environ["LANGCHAIN_API_KEY"] = "langchainKey"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"] = "educosysGenAIClass"

In [6]:
# WebBaseLoader is a document loader that can be used to load documents from the web. It takes a list of web paths (URLs) and loads the content of those URLs as documents.
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(web_paths=["https://www.educosys.com/course/genai"])
docs = loader.load()   # Load the content of the specified web page(s) into a list of documents
print(docs)

c:\conda\envs\lc312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://www.educosys.com/course/genai', 'title': 'Hands-on Generative AI Course', 'description': 'Learn, Build, Deploy and Apply Generative AI', 'language': 'en'}, page_content="Hands-on Generative AI CourseCoursesBundle CoursesMentorFree ContentTestimonialsFAQLogin Signup Starts on 6th Feb, 2026Hands-on Generative AI CourseLearn, Build, Deploy and Apply Generative AI7 weeks · Fri, Sat and Sun · 9:00 PM - 11:00 PM IST (every class) + Post-class doubt supportLast fully LIVE batch. All future courses will follow a hybrid format!Access all Live BatchesLifetime access of RecordingsAccess Discord CommunityCode availableBuild ProjectsLearn Future-Ready TechEnroll 1Week 1Foundations of Generative AI Introduction to AI Mathematical Foundations for AI Probability, Statistics, and Linear Algebra Basics of Neural Networks Gradient Descent and Optimization Basics Architectures: Feedforward, RNN, and CNN Mini Project - Build a Simple Neural Network Using TensorFlow Mi

In [7]:
# Splitting the document into smaller chunks to make it easier for the model to process and understand the content. 
# This is especially important for large documents, as it allows the model to focus on smaller sections of text, 
# improving its ability to extract relevant information and answer questions accurately.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# The RecursiveCharacterTextSplitter is a specific type of text splitter that breaks down the document into smaller chunks based 
# on character count.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

split_docs = text_splitter.split_documents(docs)  # Split the loaded documents into smaller chunks using the defined text splitter

print(split_docs[0])
print(split_docs[1])
print(split_docs[2])

page_content='Hands-on Generative AI CourseCoursesBundle CoursesMentorFree ContentTestimonialsFAQLogin Signup Starts on 6th Feb, 2026Hands-on Generative AI CourseLearn, Build, Deploy and Apply Generative AI7 weeks · Fri, Sat and Sun · 9:00 PM - 11:00 PM IST (every class) + Post-class doubt supportLast fully LIVE batch. All future courses will follow a hybrid format!Access all Live BatchesLifetime access of RecordingsAccess Discord CommunityCode availableBuild ProjectsLearn Future-Ready TechEnroll 1Week 1Foundations of Generative AI Introduction to AI Mathematical Foundations for AI Probability, Statistics, and Linear Algebra Basics of Neural Networks Gradient Descent and Optimization Basics Architectures: Feedforward, RNN, and CNN Mini Project - Build a Simple Neural Network Using TensorFlow Mini Project - Train an Autoencoder on the MNIST Dataset2Week 2Deep Generative Models Discriminative and Generative models Generative Adversarial Networks (GANs) Variational Autoencoders (VAEs) Pro

In [8]:
# Storing the chunks in a vector database to enable efficient retrieval of relevant information when answering questions.
from langchain_openai import OpenAIEmbeddings       # OpenAIEmbeddings is a class that generates vector representations (embeddings) of text using OpenAI's language models.
from langchain_community.vectorstores import Chroma  # Chroma is a vector database that allows for efficient storage and retrieval of vector representations of text.

# OpenAIEmbeddings is a class that generates vector representations (embeddings) of text using OpenAI's language models.
embeddings = OpenAIEmbeddings()

# Chroma is a vector database that allows for efficient storage and retrieval of vector representations of text.
vectorstore = Chroma.from_documents(documents=split_docs, embedding=embeddings)

print(vectorstore._collection.count())  # Print the number of documents stored in the vector database


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************sBoA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [ ]:
# Retrieving specific documents from the vector database using their unique identifiers (IDs).
print(vectorstore._collection.get())

{'ids': ['65d7460b-eca3-44c2-84c7-5f6766e1e455', '8053c4c1-6b55-4e2f-acd2-ce6baddf6387', 'd8697f34-b008-45c6-b4dc-480441c60d37', 'db048650-e0cb-4fb6-8f63-363edc0a69d1', 'ea4cbd33-4034-4a85-b097-b8092ed4e8da', '29620533-1acc-4bde-bb6f-42b7ecf45e61', '5e7f64be-59d1-4e2a-9b04-39420c8af482', '3509735a-fcef-4464-b73c-62283d5d1549', '5751c31a-eda9-4d90-883e-f158f2e50107', '2992c559-d5c7-477e-a647-f12998915f1b', '9c86ac26-75a7-4ec9-8e18-98d6d80e8bd6', 'fdde5f4f-1194-4af6-a659-f8084be50a72', '3d3d9f5f-a714-4eba-83d3-3dc4aedd590c', '13ebf8e3-6d64-489a-ac99-3a54aad8bf1b', '0b3f01cc-c2ab-4f7d-86d8-da298da0fa1a', '3e292aea-3a8e-4e21-a35a-1ea44619e8b8', 'db4f2d83-f851-4888-bbb3-5421e6a92f9d', 'd2815a17-3041-4529-9f0e-a2d1695dba86', '20e7c9e9-ff51-457d-abdd-0ecb135b902b', 'ec7ed7e6-1d4c-4b6e-a899-26926b29ee44', 'fe5fca21-e856-4512-8f51-1a90ca823922', 'cfe99009-0c4d-421b-8b8c-9ef790736756', 'c8d64262-a42b-459d-85fa-a7a3fbac6a23', '8959d3d8-91e1-4dee-adbf-92b4058475a1', 'be47833e-db32-47f2-8577-222808

In [ ]:
# Retrieving specific documents from the vector database using their unique identifiers (IDs).
print('\nCollection 1 - ',vectorstore._collection.get(ids = ['65d7460b-eca3-44c2-84c7-5f6766e1e455']))
print('\nCollection 1 - ',vectorstore._collection.get(ids = ['8053c4c1-6b55-4e2f-acd2-ce6baddf6387']))
print('\nCollection 1 - ',vectorstore._collection.get(ids = ['d8697f34-b008-45c6-b4dc-480441c60d37']))
print('\nCollection 1 - ',vectorstore._collection.get(ids = ['db048650-e0cb-4fb6-8f63-363edc0a69d1']))


Collection 1 -  {'ids': ['65d7460b-eca3-44c2-84c7-5f6766e1e455'], 'embeddings': None, 'documents': ['Hands-on Generative AI CourseCoursesBundle CoursesMentorFree ContentTestimonialsFAQLogin Signup Starts on 6th Feb, 2026Hands-on Generative AI CourseLearn, Build, Deploy and Apply Generative AI7 weeks · Fri, Sat and Sun · 9:00 PM - 11:00 PM IST (every class) + Post-class doubt supportLast fully LIVE batch. All future courses will follow a hybrid format!Access all Live BatchesLifetime access of RecordingsAccess Discord CommunityCode availableBuild ProjectsLearn Future-Ready TechEnroll 1Week 1Foundations of Generative AI Introduction to AI Mathematical Foundations for AI Probability, Statistics, and Linear Algebra Basics of Neural Networks Gradient Descent and Optimization Basics Architectures: Feedforward, RNN, and CNN Mini Project - Build a Simple Neural Network Using TensorFlow Mini Project - Train an Autoencoder on the MNIST Dataset2Week 2Deep Generative Models Discriminative and Gene

In [ ]:
# Retrieving specific documents from the vector database using their unique identifiers (IDs).
retriever = vectorstore.as_retriever() # retriever is a component that retrieves relevant information from the vector database based on a given query or question.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langsmith import Client

client = Client()

# Retrieving a prompt from the LangSmith platform using the pull_prompt method of the Client class. The prompt
prompt = client.pull_prompt("rlm/rag-prompt")  # The pull_prompt method takes the name of the prompt as an argument and retrieves the corresponding prompt from the LangSmith platform.
print(prompt)


input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


In [ ]:
from langchain_openai import ChatOpenAI

# # Initialize the language model with the specified model name and temperature setting.
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# ChatOpenAI(Chat Model) is a class that provides an interface to interact with OpenAI's chat-based language models,
# allowing for conversational interactions and responses based on the provided prompts and retrieved information.

In [ ]:
# Start creating RAG chain
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# format_docs is a function that takes a list of documents and formats them into a single string by concatenating 
# the page content of each document with two newline characters in between. This formatted string can then be used as 
# input for the language model to generate responses based on the retrieved information.

# 1. Query Fired
# 2. Related Docs from documents gets concatenated and then passed it to LLM
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# TODO: Why do we need to use RunnablePassthrough() here?
# RunnablePassthrough() is used here to indicate that the "question" input should be passed through as-is without 
# any modification or processing.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} # The retriever retrieves relevant documents based on the user's question, and the format_docs function formats those retrieved documents into a single string that can be used as context for the language model to generate a response.
    | prompt   # The prompt is used to provide a template or structure for the language model to generate responses based on the retrieved information and the user's question.
    | llm      # The language model (ChatOpenAI) is used to generate responses based on the provided prompt and the retrieved information from the vector database.
    | StrOutputParser() # StrOutputParser is a class that parses the output from the language model and converts it into a string format, making it easier to read and understand the generated response.
)

In [ ]:
rag_chain.invoke('What are the timings of the genai course?')

'The GenAI course runs for 7 weeks on Fridays, Saturdays, and Sundays from 9:00 PM to 11:00 PM IST for each class.'

In [ ]:
rag_chain.invoke('Give me the curriculam for week 1 genai course')

'Week 1 of the Generative AI course covers the Foundations of Generative AI, including an introduction to AI, mathematical foundations, basics of neural networks, gradient descent, and optimization. Students will also learn about different architectures such as feedforward, RNN, and CNN, and work on mini projects involving TensorFlow and training an autoencoder on the MNIST dataset.'

In [ ]:
rag_chain.invoke('Who is the tutor?')

'The tutor is Keerti.'

# `Printing Prompt in my langchain`

In [ ]:
# Define a function to print the prompt text and return it for further processing in the chain.
def print_prompt(prompt_text):
    print("\n Prompt -",prompt_text)
    return prompt_text

# The print_prompt function is defined to print the prompt text and return it for further processing in the chain. This allows us to see the prompt being used in the RAG chain and helps with debugging and understanding the flow of information through the chain.
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt   
    | RunnableLambda(print_prompt)          # This RunnableLambda takes the output from the prompt and passes it to the print_prompt function, which prints the prompt text and returns it for further processing in the chain.
    | llm
    | StrOutputParser()
)



In [ ]:
rag_chain.invoke('Who is the tutor?')


 Prompt - messages=[HumanMessage(content="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: I reach out to you?EducosysJoin our Discord for updates, upcoming courses and exclusive offers.ConnectFeaturesCoursesBundle CoursesTestimonialsFAQLegalPrivacy PolicyTerms of Services© 2026 ITTI Tech. All rights reserved.\n\nI reach out to you?EducosysJoin our Discord for updates, upcoming courses and exclusive offers.ConnectFeaturesCoursesBundle CoursesTestimonialsFAQLegalPrivacy PolicyTerms of Services© 2026 ITTI Tech. All rights reserved.\n\nI reach out to you?EducosysJoin our Discord for updates, upcoming courses and exclusive offers.ConnectFeaturesCoursesBundle CoursesTestimonialsFAQLegalPrivacy PolicyTerms of Services© 2026 ITTI Tech. All rights reserved.\n\nso motivating us for learning by

'The tutor is Keerti.'

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}
Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)
print("Prompt template created.")

Prompt template created.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv()

# Using Gemini Flash Model
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None
)
print("LLM configured with Gemini API!")

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

question = "What is the course about?"
try:
    # Invoke the chain
    response = rag_chain.invoke(question)
    print(f"Question: {question}")
    print(f"Answer: {response}")
except Exception as e:
    print(f"Error occurred: {e}")